**Stage 2: gender-coloured skeleton overlay** (blue = male, red = female, green = unknown) on the inpainted video.

Gender per track ID comes from the hand-filled `GENDER_BY_ID` dict; the ST-GCN classifier is **not** called here (marked as future integration in the notebook).

In [ ]:
"""
Pose Overlay with Gender Colors
- Reads pose JSON from pose_estimator.py
- Overlays colored skeletons on the INPAINTED video
- Gender is assigned per person ID (hardcoded dict below)
- Blue = Male, Red = Female, Green = Unknown

HOW TO USE:
  1. Run pose_estimator.py first — it prints all person IDs at the end
  2. Fill in GENDER_BY_ID below with those IDs
  3. Run this script

STGCN INTEGRATION (future):
  Replace GENDER_BY_ID values with output from your ST-GCN gait classifier.
  The dict key is person_id (int), value is 'male' / 'female' / 'unknown'.
"""

import cv2
import numpy as np
import json

# ── Paths ─────────────────────────────────────────────────────────────────────
pose_json_path        = '/content/vid9-pose_estimations.json'     # from pose_estimator.py
inpainted_video_path  = '/content/vid9-output_inpainted.mp4'      # from multi_object_inpainting.py
output_video_path     = '/content/vid9-pose_gender_overlay.mp4'
# ─────────────────────────────────────────────────────────────────────────────

# ══════════════════════════════════════════════════════════════════════════════
#  GENDER ASSIGNMENT — edit this dict with person IDs from pose_estimator.py
#  Keys   : person_id (int) as printed at end of pose_estimator.py
#  Values : 'male' | 'female' | 'unknown'
#
#  Example: if pose_estimator printed "Total unique person IDs: 0 to 3"
#           you'd fill in IDs 0, 1, 2, 3
# ══════════════════════════════════════════════════════════════════════════════
GENDER_BY_ID = {
    0: 'female',      # ← replace with ST-GCN output or manual assignment
    1: 'male',
    2: 'female',
    3: 'male',
    4: 'female',
    5: 'unknown',
    # add more IDs as needed
}
# ══════════════════════════════════════════════════════════════════════════════

# Colors (BGR)
GENDER_COLORS = {
    'male':    (255, 0,   0),    # Blue
    'female':  (0,   0,   255),  # Red
    'unknown': (0,   255, 0),    # Green
}

CONF_THRESHOLD = 0.5

# 17-joint COCO skeleton (matches pose_estimator.py output)
SKELETON_17 = [
    (0, 1), (0, 2),
    (1, 3), (2, 4),
    (5, 6),
    (5, 7), (7, 9),
    (6, 8), (8, 10),
    (5, 11), (6, 12),
    (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16),
]


def get_gender_color(person_id):
    """Return BGR color for person based on gender assignment."""
    gender = GENDER_BY_ID.get(person_id, 'unknown')
    return GENDER_COLORS[gender], gender


def draw_skeleton(frame, keypoints, color, conf_thresh=0.5,
                  point_radius=6, line_thickness=3):
    """Draw skeleton with white outline for visibility."""
    # Bones first (drawn under joints)
    for p1, p2 in SKELETON_17:
        if p1 >= len(keypoints) or p2 >= len(keypoints):
            continue
        k1, k2 = keypoints[p1], keypoints[p2]
        if k1['score'] > conf_thresh and k2['score'] > conf_thresh:
            pt1 = (int(k1['x']), int(k1['y']))
            pt2 = (int(k2['x']), int(k2['y']))
            cv2.line(frame, pt1, pt2, (255, 255, 255), line_thickness + 2, cv2.LINE_AA)
            cv2.line(frame, pt1, pt2, color,           line_thickness,     cv2.LINE_AA)

    # Joints on top
    for kp in keypoints:
        if kp['score'] > conf_thresh:
            center = (int(kp['x']), int(kp['y']))
            cv2.circle(frame, center, point_radius + 2, (255, 255, 255), -1)
            cv2.circle(frame, center, point_radius,     color,           -1)


def draw_gender_label(frame, keypoints, person_id, gender, color, conf_thresh=0.5):
    """Draw person ID + gender label above the person."""
    valid = [kp for kp in keypoints if kp['score'] > conf_thresh]
    if not valid:
        return
    top_y = int(min(kp['y'] for kp in valid)) - 15
    top_x = int(np.mean([kp['x'] for kp in valid]))
    label = f"ID:{person_id} {gender}"

    # Shadow for readability
    cv2.putText(frame, label, (top_x - 2, max(top_y, 20) + 2),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 0), 3)
    cv2.putText(frame, label, (top_x - 2, max(top_y, 20)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)


# ── Load pose JSON ────────────────────────────────────────────────────────────
print("Loading pose estimations...")
with open(pose_json_path, 'r') as f:
    all_frames_data = json.load(f)
print(f"✓ Loaded {len(all_frames_data)} frames of pose data")

# ── Open inpainted video ──────────────────────────────────────────────────────
cap = cv2.VideoCapture(inpainted_video_path)
if not cap.isOpened():
    raise IOError(f"Cannot open inpainted video: {inpainted_video_path}")

frame_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps          = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Inpainted video: {frame_width}x{frame_height}, FPS={fps}, Frames={total_frames}")

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

# ── Draw legend in first frame area ──────────────────────────────────────────
def draw_legend(frame):
    """Draw color legend in top-right corner."""
    legends = [('Male',    GENDER_COLORS['male']),
               ('Female',  GENDER_COLORS['female']),
               ('Unknown', GENDER_COLORS['unknown'])]
    x_start = frame_width - 130
    for i, (label, color) in enumerate(legends):
        y = 25 + i * 28
        cv2.rectangle(frame, (x_start, y - 14), (x_start + 18, y + 2), color, -1)
        cv2.rectangle(frame, (x_start, y - 14), (x_start + 18, y + 2), (255,255,255), 1)
        cv2.putText(frame, label, (x_start + 24, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, label, (x_start + 24, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1)

# ── Process frames ────────────────────────────────────────────────────────────
print("Overlaying gender-colored skeletons...")
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx < len(all_frames_data):
        frame_persons = all_frames_data[frame_idx]

        for person in frame_persons:
            pid       = person.get('person_id', -1)
            keypoints = person['keypoints']
            color, gender = get_gender_color(pid)

            draw_skeleton(frame, keypoints, color, CONF_THRESHOLD)
            draw_gender_label(frame, keypoints, pid, gender, color, CONF_THRESHOLD)

    draw_legend(frame)
    writer.write(frame)
    frame_idx += 1

    if frame_idx % 25 == 0:
        print(f"  {frame_idx}/{total_frames} frames done")

cap.release()
writer.release()

print(f"\n✓ Output saved → {output_video_path}")
print("\nGender assignments used:")
for pid, gender in GENDER_BY_ID.items():
    color_name = {'male': 'Blue', 'female': 'Red', 'unknown': 'Green'}[gender]
    print(f"  Person ID {pid}: {gender} ({color_name})")

from IPython.display import Video
Video(output_video_path, embed=True, width=720)